# Speech enhancement and separation with ESPnet-SE

Run a pretrained enhancement model on real noisy speech, separate a
two-speaker mixture, and score both with VERSA and a pretrained ASR model.

This is the demonstration part of an assignment from CMU 11492/11692/18495,
*Speech Technology for Conversational AI*, kept here without the graded
exercises so that it runs end to end.

Main references:
- [ESPnet repository](https://github.com/espnet/espnet)
- [ESPnet documentation](https://espnet.github.io/espnet/)
- [ESPnet-SE recipe template](https://github.com/espnet/espnet/tree/master/egs2/TEMPLATE/enh1)
- [VERSA](https://github.com/wavlab-speech/versa)


# Contents

1. Install
2. Speech enhancement with a pretrained model
3. Speech separation
4. Evaluating separated speech with a pretrained ASR model


## Install

We use the inference install of ESPnet rather than the full one: it carries no
training stack, so it is much smaller and much faster to install. VERSA is the
evaluation toolkit used further down.


In [ ]:
import locale
locale.getpreferredencoding = lambda *a, **kw: "UTF-8"

# [enh] brings fast_bss_eval, which the enhancement losses need at load time;
# editdistance is used by the word error rate comparison further down
%pip install -q "espnet[enh]==202610.post1"
%pip install -q editdistance


In [ ]:
!rm -rf versa
!git clone https://github.com/wavlab-speech/versa
!cd versa && pip install .
!cd ..
!pip install librosa==0.9.2 numpy==1.23.5
!pip install numpy==1.23.5 tensorflow==2.12.0 jax==0.4.9 jaxlib==0.4.9 ml_dtypes==0.2.0 transformers
# IMPORTANT NOTE: due to the recent default change in colab, we need to restart the session to make the numpy(1.23.5) work as expected
# Go to `Runtime` and select `Restart Session`

## Speech Enhancement with Pretrained Models


### Single-Channel Enhancement


### Task1  )

Run inference of pretrained enhancement model.


In [ ]:
# Download one utterance from real noisy speech of CHiME4
!gdown 1SmrN5NFSg6JuQSs2sfy3ehD8OIcqK6wS -O M05_440C0213_PED_REAL.wav
import os

import soundfile
from IPython.display import display, Audio
mixwav_mc, sr = soundfile.read("M05_440C0213_PED_REAL.wav")
# mixwav.shape: num_samples, num_channels
mixwav_sc = mixwav_mc[:,4]
display(Audio(mixwav_mc.T, rate=sr))

#### Download and load the pretrained [Universal Speech Enhancement](https://arxiv.org/abs/2309.17384)


In [ ]:
import torch
import soundfile
from espnet2.bin.enh_inference import SeparateSpeech

# from_pretrained fetches the model from the Hub and caches it. Cloning the
# repository with git instead needs git-lfs, and without it the checkpoint
# arrives as a text pointer and torch.load fails on it.
enh_model_sc = SeparateSpeech.from_pretrained(
    model_tag="espnet/Wangyou_Zhang_universal_train_enh_uses_refch0_2mem_raw",
    # for segment-wise process on long speech
    normalize_segment_scale=False,
    show_progressbar=True,
    ref_channel=4,
    normalize_output_wav=True,
    device="cuda" if torch.cuda.is_available() else "cpu" if torch.cuda.is_available() else "cpu",
)


#### Enhance the single-channel real noisy speech in CHiME4


In [ ]:
# play the enhanced single-channel speech
wave = enh_model_sc(mixwav_sc[None, ...], sr)

print("Input real noisy speech", flush=True)
display(Audio(mixwav_sc, rate=sr))
print("Enhanced speech", flush=True)
display(Audio(wave[0].squeeze(), rate=sr))

In [ ]:
import numpy as np
soundfile.write("enhanced_speech.wav", np.ravel(wave[0].squeeze()), sr)

In [ ]:
with open("test_wav.scp", "w") as test_wavscp:

  print(
      "enhanced_speech_eg enhanced_speech.wav", file = test_wavscp, end=""
  )


In [ ]:
with open("versa_metrics_se.yaml", 'w') as out_f:
  content = """\n
    - name: pseudo_mos
      predictor_types: ["utmos", "dnsmos", "plcmos"]
      predictor_args:
        utmos:
          fs: 16000
        dnsmos:
          fs: 16000
        plcmos:
          fs: 16000
        singmos:
          fs: 16000
    - name: squim_no_ref"""
  out_f.write(content)

In [ ]:
!python -m versa.bin.scorer \
    --score_config versa_metrics_se.yaml \
    --pred test_wav.scp \
    --output_file test_result

In [ ]:
!cat test_result

#### Portable speech enhancement scripts for other tasks

For an ESPNet ASR or TTS dataset like below:

```
data
`-- et05_real_isolated_6ch_track
    |-- spk2utt
    |-- text
    |-- utt2spk
    |-- utt2uniq
    `-- wav.scp
```

Run the following scripts to create an enhanced dataset:

```
scripts/utils/enhance_dataset.sh \
    --spk_num 1 \
    --gpu_inference true \
    --inference_nj 4 \
    --fs 16k \
    --id_prefix "" \
    dump/raw/et05_real_isolated_6ch_track \
    data/et05_real_isolated_6ch_track_enh \
    exp/enh_train_enh_beamformer_mvdr_raw/valid.loss.best.pth
```

The above script will generate a new directory data/et05_real_isolated_6ch_track_enh:

```
data
`-- et05_real_isolated_6ch_track_enh
    |-- spk2utt
    |-- text
    |-- utt2spk
    |-- utt2uniq
    |-- wav.scp
    `-- wavs/
```
where wav.scp contains paths to the enhanced audios (stored in wavs/).


## Speech Separation


### Model selection

The separation models trained on wsj0_2mix are published in the [ESPnet
organisation](https://huggingface.co/espnet) and load by tag. The course used
a TF-GridNet checkpoint kept on Google Drive; the Hub copy of that one is an
empty repository, so this uses DPTNet instead, which is published properly and
separates the same mixtures.


In [ ]:
import torch
import soundfile
from espnet2.bin.enh_inference import SeparateSpeech

separate_speech = SeparateSpeech.from_pretrained(
    model_tag="espnet/Wangyou_Zhang_wsj0_2mix_enh_train_enh_dptnet_raw",
    # for segment-wise process on long speech
    segment_size=2.4,
    hop_size=0.8,
    normalize_segment_scale=False,
    show_progressbar=True,
    ref_channel=None,
    normalize_output_wav=True,
    device="cuda" if torch.cuda.is_available() else "cpu" if torch.cuda.is_available() else "cpu",
)


### Separate Speech Mixture


#### Separate the example in wsj0_2mix testing set


### Task2  )

Run inference of pretrained speech seperation model based on TF-GRIDNET.


In [ ]:
!gdown 1ZCUkd_Lb7pO2rpPr4FqYdtJBZ7JMiInx -O 447c020t_1.2106_422a0112_-1.2106.wav

import os
import soundfile
from IPython.display import display, Audio

mixwav, sr = soundfile.read("447c020t_1.2106_422a0112_-1.2106.wav")
waves_wsj = separate_speech(mixwav[None, ...], fs=sr)

print("Input mixture", flush=True)
display(Audio(mixwav, rate=sr))
print(f"========= Separated speech with model =========", flush=True)
print("Separated spk1", flush=True)
display(Audio(waves_wsj[0].squeeze(), rate=sr))
print("Separated spk2", flush=True)
display(Audio(waves_wsj[1].squeeze(), rate=sr))

#### Show spectrums of separated speech


Show wavform and spectrogram of mixed and seperated speech.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

from espnet2.layers.stft import Stft

stft = Stft(n_fft=512, win_length=None, hop_length=128, window="hann")
ilens = torch.LongTensor([len(mixwav)])


def magnitude(wav):
    """|STFT| of one waveform, as (frequency, time)."""
    spec = stft(torch.as_tensor(wav).reshape(1, -1), ilens)[0].squeeze()
    # the last dimension holds (real, imaginary)
    return torch.linalg.norm(spec, dim=-1).transpose(-1, -2).numpy()


def plot_spectrogram(ax, wav, title):
    mag = magnitude(wav)
    ax.imshow(
        20 * np.log10(np.maximum(mag, 1e-8)),
        origin="lower",
        aspect="auto",
        extent=[0, len(mixwav) / sr, 0, sr / 2],
        cmap="magma",
    )
    ax.set_title(title)
    ax.set_ylabel("Frequency (Hz)")


def plot_waveform(ax, wav, title):
    ax.plot(torch.linspace(0, len(mixwav) / sr, len(mixwav)), np.ravel(wav))
    ax.set_xlim(0, len(mixwav) / sr)
    ax.set_title(title)


fig, axes = plt.subplots(3, 2, figsize=(18, 9))
for row, (wav, name) in enumerate(
    [(mixwav, "Mixture"), (waves_wsj[0], "Separated (spk1)"),
     (waves_wsj[1], "Separated (spk2)")]
):
    plot_spectrogram(axes[row][0], wav, f"{name} spectrogram")
    plot_waveform(axes[row][1], wav, f"{name} waveform")
axes[2][1].set_xlabel("Time (s)")
plt.tight_layout()
plt.show()


## Evaluate separated speech with pretrained ASR model

The ground truths are:

`text_1: SOME CRITICS INCLUDING HIGH REAGAN ADMINISTRATION OFFICIALS ARE RAISING THE ALARM THAT THE FED'S POLICY IS TOO TIGHT AND COULD CAUSE A RECESSION NEXT YEAR`

`text_2: THE UNITED STATES UNDERTOOK TO DEFEND WESTERN EUROPE AGAINST SOVIET ATTACK`

(This may take a while for the speech recognition.)


In [ ]:
%pip install -q https://github.com/kpu/kenlm/archive/master.zip # ASR needs kenlm

### Task3  )

Show inference of pre-trained ASR model (trained on WSJ corpus) on mixed and seperated speech.


In [ ]:
import torch
from espnet2.bin.asr_inference import Speech2Text

# a WSJ model from the organisation, so nothing here depends on a Google
# Drive copy staying up
speech2text = Speech2Text.from_pretrained(
    model_tag="espnet/kamo-naoyuki_wsj_transformer2",
    ngram_weight=0.0,
    lm_weight=0.0,
    device="cuda" if torch.cuda.is_available() else "cpu" if torch.cuda.is_available() else "cpu",
)

text_est = [None, None]
text_est[0], *_ = speech2text(waves_wsj[0].squeeze())[0]
text_est[1], *_ = speech2text(waves_wsj[1].squeeze())[0]
text_m, *_ = speech2text(mixwav)[0]
print("Mix Speech to Text: ", text_m)
print("Separated Speech 1 to Text: ", text_est[0])
print("Separated Speech 2 to Text: ", text_est[1])


In [ ]:
import difflib
from itertools import permutations

import editdistance
import numpy as np

colors = dict(
    red=lambda text: f"\033[38;2;255;0;0m{text}\033[0m" if text else "",
    green=lambda text: f"\033[38;2;0;255;0m{text}\033[0m" if text else "",
    yellow=lambda text: f"\033[38;2;225;225;0m{text}\033[0m" if text else "",
    white=lambda text: f"\033[38;2;255;255;255m{text}\033[0m" if text else "",
    black=lambda text: f"\033[38;2;0;0;0m{text}\033[0m" if text else "",
)

def diff_strings(ref, est):
    """Reference: https://stackoverflow.com/a/64404008/7384873"""
    ref_str, est_str, err_str = [], [], []
    matcher = difflib.SequenceMatcher(None, ref, est)
    for opcode, a0, a1, b0, b1 in matcher.get_opcodes():
        if opcode == "equal":
            txt = ref[a0:a1]
            ref_str.append(txt)
            est_str.append(txt)
            err_str.append(" " * (a1 - a0))
        elif opcode == "insert":
            ref_str.append("*" * (b1 - b0))
            est_str.append(colors["green"](est[b0:b1]))
            err_str.append(colors["black"]("I" * (b1 - b0)))
        elif opcode == "delete":
            ref_str.append(ref[a0:a1])
            est_str.append(colors["red"]("*" * (a1 - a0)))
            err_str.append(colors["black"]("D" * (a1 - a0)))
        elif opcode == "replace":
            diff = a1 - a0 - b1 + b0
            if diff >= 0:
                txt_ref = ref[a0:a1]
                txt_est = colors["yellow"](est[b0:b1]) + colors["red"]("*" * diff)
                txt_err = "S" * (b1 - b0) + "D" * diff
            elif diff < 0:
                txt_ref = ref[a0:a1] + "*" * -diff
                txt_est = colors["yellow"](est[b0:b1]) + colors["green"]("*" * -diff)
                txt_err = "S" * (b1 - b0) + "I" * -diff

            ref_str.append(txt_ref)
            est_str.append(txt_est)
            err_str.append(colors["black"](txt_err))
    return "".join(ref_str), "".join(est_str), "".join(err_str)


text_ref = [
  "SOME CRITICS INCLUDING HIGH REAGAN ADMINISTRATION OFFICIALS ARE RAISING THE ALARM THAT THE FED'S POLICY IS TOO TIGHT AND COULD CAUSE A RECESSION NEXT YEAR",
  "THE UNITED STATES UNDERTOOK TO DEFEND WESTERN EUROPE AGAINST SOVIET ATTACK",
]

print("=====================" , flush=True)
perms = list(permutations(range(2)))
string_edit = [
  [
    editdistance.eval(text_ref[m], text_est[n])
    for m, n in enumerate(p)
  ]
  for p in perms
]

dist = [sum(edist) for edist in string_edit]
perm_idx = np.argmin(dist)
perm = perms[perm_idx]

for i, p in enumerate(perm):
  print("\n--------------- Text %d ---------------" % (i + 1), flush=True)
  ref, est, err = diff_strings(text_ref[i], text_est[p])
  print("REF: " + ref + "\n" + "HYP: " + est + "\n" + "ERR: " + err, flush=True)
  print("Edit Distance = {}\n".format(string_edit[perm_idx][i]), flush=True)